# 1D OptimalFilterPhonon

The following code takes three arrays:
1. template
2. pulse
3. noise signal

and performs an optimization on two free parameters:
1. amplitude ($\hat{A}$)
2. time-shift ($\hat{t}$)

We want to best estimate the ratio of amplitudes between a template and a pulse trace. The solution is to perform frequency-domain $\chi^2$ minimization using the Fourier transforms of the trace, template and noise.


Express the signal in frequency space as $\tilde{v}(f) = \hat{A} \cdot \tilde{s}(f) + n(f)$


For more information, please see the [SuperCDMS Processing Documentation](https://supercdms.gitlab.io/Reconstruction/cdmsbats/BatCommon/OF/)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import ROOT 
from cats.cdataframe import CDataFrame

## Setup

In [ ]:
# define file paths to the template file, PSDfile, and DMCfile
Templatefile = 'Templates_8192.pkl'
PSDfile = 'PSDs_27190502_170014_0.joblib'
DMCfile = 'supersim_test_energyGap_10220626_0000.root'

In [ ]:
# obtain template. Contains "CH3", "CH2", and "Total". Our studies only use CH3.

import pickle
import joblib

Templates=joblib.load(Templatefile) 

# obtain PSD (should be 'ASD'). Contains "CH3", "CH2", and "Total". Our studies only use CH3.

org_PSD_data=joblib.load(PSDfile)
PSD = org_PSD_data['PSD']['CH3'] # Amplitude of PSD, given in pA/sqrt(Hz)
frequency = org_PSD_data['f'] # Frequencies of PSD, given in Hz

PSD *= 1e-6 * np.sqrt(frequency) # pico amps / sqrt(Hz) -> micro amps

In [ ]:
# obtain and show the source macro given the root file path
def getMacro(DMCfile):
    f=TFile.Open(DMCfile)
    macro=f.Get("G4SettingsInfoDir/SuperSim_Macro")
    macro.Print()

In [ ]:
# obtain and show the software versions given the root file path
def getVersions(DMCfile):
    print(CDataFrame("G4SettingsInfoDir/Versions", DMCfile).AsNumpy())

In [ ]:
def getTES(DMCfile):
    #### Retrieving the TES traces for each channel ####
    g4dmcTES = CDataFrame("G4SimDir/g4dmcTES", DMCfile).Filter('DataType==0')
    Trace = g4dmcTES.Filter('ChanNum==0').AsNumpy(['Trace'])['Trace'][0] # Save only the trace from the inner channel
    Trace = max(Trace) - Trace # flip traces right-side up

    # Let's also get the starting time of the TES pulse T0 and the width of the timebins BinWidth. 
    # We can use this to set up the time array TimeBins in units of microseconds.
    T0       = g4dmcTES.AsNumpy(['T0'])['T0'][0]
    BinWidth = g4dmcTES.AsNumpy(['BinWidth'])['BinWidth'][0]
    TimeBins = np.arange(T0, T0 + BinWidth * len(Trace), BinWidth) * 1e-3 # ns -> us
    
    return TimeBins, Trace

In [ ]:
# obtaining needed data for optimal filter
TimeBins, Trace = getTES(DMCfile)

## Interpolate the PSD for the Required Frequency Bins

In [ ]:
#### Fourier Transform of Time Domain ####

from scipy.fft import fft, fftfreq

N, d = 8191, 0.8e-6 # number of discrete values, time between them (s)
xf = fftfreq(N, d) # frequencies for fft

#### Interpolate PSD ####

from scipy.interpolate import interp1d

PSD_approx = interp1d(frequency, PSD, kind='cubic') # approximation of PSD

cut = xf >= 0
plt.figure(figsize=(4,3), dpi = 200)
plt.plot(frequency, PSD, label = 'Original PSD', color = 'slategray') # plot of original PSD vs. original frequencies
plt.plot(xf[cut], PSD_approx(xf[cut]), label = 'Interpolated PSD', color = 'black') # plot of interpolated PSD
plt.legend()
plt.xlabel('Frequency [Hz]')
plt.ylabel(r'Amplitude [$\mu$A]')
plt.title('PSD')

# The original PSD only has amplitudes for positive frequencies.
# We'll need to interpolate for our needed positive frequencies and then add a mirror copy to the array
# for the negative frequencies.

yf_psd = PSD_approx(xf[cut])
n = np.concatenate((yf_psd, (yf_psd[1:])[::-1])) # array of noise signal for positive and then negative frequencies

## Template, Pulse, and PSD in Frequency Space

In [ ]:
#### Fourier Transform of Simulated Pulse and Template ####

from scipy.fft import fft, fftfreq

trace = Trace[8194: 8194 + 8191] # channel 1 TESSim pulse
template = Templates['CH3'][1:] # channel 1 template

N, d = 8191, 0.8e-6    # number of discrete values, time between them (s)
xf = fftfreq(N, d)    # produce frequencies for fft (the positive frequencies come before the negative)
S = fft(template)    # fourier transform of CH1 template
V = fft(trace)      # fourier transform of CH1 trace

J = n**2    # J = <n^2> where n is the interpolated PSD with negative and positive frequencies
J[J == 0] = np.inf    # avoid 'divide by zero' errors due to J

cut = xf > 0
plt.figure(figsize=(4,3), dpi = 200)
plt.scatter(xf[cut], np.abs(V[cut]) / np.sqrt(xf[cut]), s = 1, label='TES trace', color='lightsteelblue')
plt.scatter(xf[cut], np.abs(S[cut]) / np.sqrt(xf[cut]), s = 1, label = 'Template',color='black')
plt.scatter(xf[cut], n[cut] / np.sqrt(xf[cut]), s = 1, label = 'Interpolated PSD',color='slategray')
plt.xscale('log')
plt.yscale('log')
plt.ylim(1e-10, 1e2)
plt.legend()
plt.xlabel('Frequency [Hz]')
plt.ylabel(r'Amplitude [$\mu$A / $\sqrt{\mathrm{Hz}}$]')
plt.title('Amplitude vs. Frequency')

## Amplitude vs. Time Shift
# $\hat{A} = \frac{\Sigma e^{i \omega \hat{t}} \tilde{s}^*_k \tilde{v}_k / J_k}{\Sigma |\tilde{s}_k|^2 / J_k}$

Given a time shift ($t$) between a pulse and the template, the optimal filter finds an amplitude ($A$) that minimizes the $\chi^2$ between the two. 
<br>The optimal timeshift ($\hat{t}$) produces the maximum amplitude ($\hat{A}$).

The segment of code below does a scan of time shifts and reports $\hat{t}$ and $\hat{A}$.

In [ ]:
#### Fourier Transform of Simulated Pulse and Template ####

from scipy.fft import fft, fftfreq

trace = Trace[8194: 8194 + 8191] # channel 1 TESSim pulse
template = Templates['CH3'][1:] # channel 1 template

N, d = 8191, 0.8e-6    # number of discrete values, time between them (s)
xf = fftfreq(N, d)    # produce frequencies for fft (the positive frequencies come before the negative)
S = fft(template)    # fourier transform of CH1 template
V = fft(trace)      # fourier transform of CH1 trace
J = n**2           # J = <n^2> where n is the interpolated PSD with negative and positive frequencies
J[J == 0] = np.inf    # avoid 'divide by zero' errors due to J
w = 2 * np.pi * xf    # Convert linear to angular frequency


# Plot of Amplitude as a function of time shift.
plt.figure(figsize=(6,4), dpi = 200)

timescan = np.arange(-100 * 0.8e-6, 101 * 0.8e-6, 0.8e-6) #scan from -100 bins to 100 bins

### Use the equation of for A above. The maximum is \hat{A} and the t that gives the maximum is \hat{t} ####
amplitude = np.array([sum(np.conj(S) * V * np.exp(1j * w * t) / J) / sum(np.abs(S)**2 / J) for t in timescan])
plt.scatter(timescan, np.abs(amplitude), color = 'lightsteelblue')

A_hat = max(np.abs(amplitude))
t_hat = timescan[np.abs(amplitude).tolist().index(A_hat)]

props = dict(boxstyle='square', facecolor='white', alpha=1)
plt.axvline(t_hat, 0, 1, lw = 2, color = 'black', ls = '--')
plt.text(t_hat, 0.85 * A_hat,  r'$\hat{\mathrm{t}}$ = ' + f'{np.round(t_hat*1e6, 1)}' + r' $\mu$s or ' + 
         f'{round(t_hat/(0.8e-6))} bins' + '\n' + r'$\hat{\mathrm{A}}$ = ' + f'{A_hat:.3e}', 
         ha = 'center', va = 'center', bbox = props, fontsize = 10)

plt.xlabel(r'Time Shift of TES Pulse [s]')
plt.ticklabel_format(axis="x", style="sci", scilimits=(0,0))
plt.xlim(-100 * 0.8e-6, 101 * 0.8e-6)
plt.ylabel(r'Amplitude [$\mu$A]')
plt.title('Amplitude vs Time Shift of TES Pulse')

### Using the IFFT Trick to Derive Best Amplitude Directly

In [ ]:
from scipy.fft import fft, fftfreq, ifft

trace = Trace[8194: 8194 + 8191] # channel 1 TESSim pulse
template = Templates['CH3'][1:] # channel 1 template

N, d = 8191, 0.8e-6    # number of discrete values, time between them (s)
xf = fftfreq(N, d)    # produce frequencies for fft (the positive frequencies come before the negative)
S = fft(template)    # fourier transform of CH1 template
V = fft(trace)      # fourier transform of CH1 trace
J = n**2           # J = <n^2> where n is the interpolated PSD with negative and positive frequencies
J[J == 0] = np.inf    # avoid 'divide by zero' errors due to J

stn = sum(np.conj(S)*S / J) # signal-to-noise ratio

amp0 = sum(np.conj(S) * V / J) / stn # amplitude at 0 time delay
ampfunc_freq = (np.conj(S) * V / J) / stn # amplitude as a function of frequency bin
ampfunc = ifft(ampfunc_freq) # inverse-FFT to retrieve amplitudes as a function of time delay

maxamp = ampfunc.real.max(axis=0)*N
print(f'The maximum amplitude is {maxamp} uA')
tdelay = ampfunc.real.argmax(axis=0)
print(f'The time delay corresponding to the maximum amplitude is {tdelay-N} bins')

## Chi-Square vs. Time Shift
# $\chi^2 = \int_{-\infty}^{\infty} df \frac{|\tilde{v}(f) - Ae^{-i \omega t} \tilde{s}(f)|^2}{J(f)}$

Given a time shift ($t$) between a pulse and the template, the optimal filter finds an amplitude ($A$) that minimizes the $\chi^2$ between the two. 
<br>The optimal timeshift ($\hat{t}$) produces the minimum $\chi^2$.

Setting $A$ to $\hat{A}$ found from the Amplitude vs. Time-Shift scan, the segment of code below does another scan of time shifts and confirms that $\hat{t}$ gives the minimum $\chi^2$.

In [ ]:
#### Fourier Transform of Simulated Pulse and Template ####

from scipy.fft import fft, fftfreq

trace = Trace[8194: 8194 + 8191] # channel 1 TESSim pulse
template = Templates['CH3'][1:] # channel 1 template

N, d = 8191, 0.8e-6    # number of discrete values, time between them (s)
xf = fftfreq(N, d)    # produce frequencies for fft (the positive frequencies come before the negative)
S = fft(template)    # fourier transform of CH1 template
V = fft(trace)      # fourier transform of CH1 trace
J = n**2           # J = <n^2> where n is the interpolated PSD with negative and positive frequencies
J[J == 0] = np.inf    # avoid 'divide by zero' errors due to J
w = 2 * np.pi * xf    # Convert linear to angular frequency


# Plot of \chi^2 as a function of time shift.
plt.figure(figsize=(6,4), dpi = 200)

timescan = np.arange(-100 * 0.8e-6, 101 * 0.8e-6, 0.8e-6) #scan from -100 bins to 100 bins

### Use the equation of for \chi^2 above. Set A to \hat{A}. The t that gives the minimum is \hat{t} ####
chisquare = np.array([sum(np.abs(V - A_hat * np.exp(-1j * w * t) * S) ** 2 / J) for t in timescan])
plt.scatter(timescan, chisquare, color = 'lightsteelblue')

chi2 = min(chisquare)
t_hat = timescan[chisquare.tolist().index(chi2)]

props = dict(boxstyle='square', facecolor='white', alpha=1)
plt.axvline(t_hat, 0, 1, lw = 2, color = 'black', ls = '--')
plt.text(t_hat, 1.5e3 * chi2,  r'$\hat{\mathrm{t}}$ = ' + f'{np.round(t_hat*1e6, 1)}' + f' $\mu$s or ' + 
         f'{round(t_hat/(0.8e-6))} bins' + '\n' + r'$\hat{\mathrm{A}}$ = ' + f'{A_hat:.3e}' + '\n'
         r'$\chi^2$ = ' + f'{chi2:.3e}', 
         ha = 'center', va = 'center', bbox = props, fontsize = 10)

plt.xlabel(r'Time Shift of TES Pulse [s]')
plt.ticklabel_format(axis="x", style="sci", scilimits=(0,0))
plt.xlim(-100 * 0.8e-6, 101 * 0.8e-6)
plt.ylabel(r'$\chi^2$')
plt.title(r'$\chi^2$ vs Time Shift of TES Pulse')

## TES trace superimposed with reconstructed pulse
The reconstructed pulse is plotted by taking the template and scaling its amplitude by $\hat{A}$ and shifting it in time by $\hat{t}$.

In [ ]:
plt.figure(figsize=(6,4), dpi = 200)

plt.plot(TimeBins[8194: 8194 + 8191], Trace[8194: 8194 + 8191], color = 'slategray', label = 'CH1 TES Trace', lw = 2)
plt.plot(TimeBins[8194: 8194 + 8191] + t_hat*1e6, Templates['CH3'][1:] * A_hat, color = 'black', ls = '--', label = 'OF Reconstructed Trace', lw = 2)

plt.xlim(-100, 2000)
plt.xlabel(r'Time [$\mathrm{\mu s}$]')
plt.ylabel(r'Amplitude [$\mathrm{\mu A}$]')
plt.legend()

In [ ]:
plt.figure(figsize=(6,4), dpi = 200)

plt.plot(TimeBins[8194: 8194 + 8191], Trace[8194: 8194 + 8191], color = 'slategray', label = 'CH1 TES Trace', lw=2)
plt.plot(TimeBins[8194: 8194 + 8191] + t_hat*1e6, Templates['CH3'][1:] * A_hat, color = 'black', ls = '--', label = 'OF Reconstructed Trace', lw=2)

plt.xlim(-100, 2000)
plt.xlabel(r'Time [$\mathrm{\mu s}$]')
plt.ylabel(r'Amplitude [$\mathrm{\mu A}$]')
plt.yscale('log')
plt.legend()